# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template and interactive code for loading and exploring the [FAIR² Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL as specified below.

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and explore the schema using `mlcroissant`. This gives a high-level overview of the dataset, its structure, and its description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Display top-level metadata
meta = dataset.metadata
print(f"Dataset title: {meta.name}")
print(f"Description: {meta.description}")
print(f"Published: {meta.datePublished if hasattr(meta, 'datePublished') else 'n/a'}")

## 2. Data Overview
Explore available Record Sets and fields as defined in the Croissant schema. All references are made via explicit `@id` identifiers for RecordSets, Fields, and Columns.

In [ ]:
# List all record sets with their @id, name, and description
print("Available Record Sets in this dataset:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}, description: {getattr(rs, 'description', '')}")

# For each record set, show its fields (columns) and their @id
for rs in record_sets:
    print(f"\nFields for RecordSet @id={rs.id}, name={rs.name}:")
    for field in rs.fields:
        print(f"  - Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', 'unknown')}")

## 3. Data Extraction
Let's extract actual records from a selected RecordSet. You must use the appropriate `@id` of the RecordSet you want to load. We'll load all record sets and collect them into pandas DataFrames.

In [ ]:
# Prepare list of RecordSet @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Available record_set @ids: {record_set_ids}")

# Create DataFrames for all record sets
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records, fields: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f"  No records found for @id={rs_id}")

# For this notebook, pick the primary RecordSet (@id) with non-empty DataFrame for demo
main_rs_id = next(iter([k for k, v in dataframes.items() if not v.empty]), None)
if main_rs_id is not None:
    df = dataframes[main_rs_id]
    print(f"\nPreview of main DataFrame from RecordSet @id: {main_rs_id}")
    display(df.head())
else:
    print("No non-empty RecordSets available to load.")

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate filtering, normalizing, and grouping operations, using explicit field (column) `@id`s referenced from the metadata. Adjust the code below to select a field (for example, numeric columns like peptide counts, abundance, or molecular weight).

> **Tip:** Refer to the field overview table above to select possible field `@id`s. Replace placeholder values with actual field `@id`s for your analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Choose a numeric field (by @id) for analysis, e.g., "MW" (Molecular Weight) or "Abundance"
# You can list all columns with: df.columns.tolist()
numeric_field_id = None
for col in df.columns:
    if ("weight" in col.lower()) or ("mw" == col.lower()) or ("count" in col.lower()) or ("abundance" in col.lower()):
        numeric_field_id = col
        print(f"Detected numeric field: {col}")
        break
if numeric_field_id is None:
    # fallback: pick first numeric-looking column
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            print(f"Fallback to numeric field: {col}")
            break

print(f"Using numeric field for EDA: {numeric_field_id}")

# Filter: select records where the value is above a threshold (example: 10)
threshold = 10
filtered_df = df.copy()
if numeric_field_id is not None and np.issubdtype(df[numeric_field_id].dtype, np.number):
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
else:
    print(f"Field {numeric_field_id} is not numeric or not found.")

print(filtered_df.head())

# Normalize (z-score) the chosen numeric field
if numeric_field_id and numeric_field_id in filtered_df.columns:
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a field (e.g., "Description" or other categorical column)
# Choose a group field by @id/name
group_field = None
for col in df.columns:
    if ("desc" in col.lower()) or ("type" in col.lower()) or ("category" in col.lower()):
        group_field = col
        print(f"Detected group field: {col}")
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found; skipping group summary.")

## 5. Visualization
Let's visualize the normalized and filtered numeric field, as well as any interesting relationships between fields.
Below is an example histogram and a scatter plot. You can modify to suit your data (e.g., change fields as appropriate).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=30)
    plt.title(f'Distribution of {numeric_field_id} after filtering')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, color="g")
        plt.title(f'Normalized distribution of {numeric_field_id} (z-score)')
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel('Count')
        plt.show()

    # Try a scatter if group_field is categorical with few levels
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the Croissant FAIR² dataset describing mass spectrometry protein abundance in human mast cell extracellular vesicles. Using only the `@id` properties for all RecordSets, Fields, and Columns, we:
- Inspected metadata, fields, and record set structure
- Loaded data dynamically into pandas DataFrames
- Filtered, normalized, and visualized numeric fields

The dataset provides a rich set of columns suitable for further downstream analysis such as biomarker discovery, differential protein expression, and more. You are encouraged to explore the field catalog for advanced analysis of specific questions relevant to your use case.